# Kaggle — chẩn đoán ATE

Notebook này **chỉ** làm ATE, mục tiêu là trả lời: F1 ~0.40 đang thua ở đâu.

Thứ tự cố ý: mọi kiểm tra **không cần train** chạy trước. Nếu pipeline
decode/normalize/metric đã hỏng thì train bao lâu cũng không cứu được, và
biết điều đó mất 30 giây thay vì 3 giờ.

| cell | câu hỏi | cần train? |
|---|---|---|
| 3 | Dữ liệu có emoji, thiếu `$T$`, aspect không khớp câu? | không |
| 4 | **Trần của pipeline**: đưa chính GOLD qua decode+normalize thì có lấy lại được gold? | không |
| 5 | Baseline từ vựng: khớp aspect của train vào câu test được bao nhiêu? | không |
| 6 | Train ATE | có |
| 7 | Model sinh ra **chữ gì**: raw → decode → normalize → gold | có |
| 8 | Phân loại lỗi và kết luận | có |

Cell 4 là quan trọng nhất. `decode_and_normalize` ánh xạ mỗi aspect về n-gram
gần nhất của câu, mà `build_ngram_vocabulary` tách câu bằng khoảng trắng —
nên nếu trong câu aspect dính dấu câu (`giá cả,`) thì n-gram là `cả,` và
ngay cả GOLD cũng bị ánh xạ sai. Cell 4 đo trực tiếp trần đó.

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/hotuyen21pt/KLTN-Token-Merging.git'
BRANCH = 'tuyen'
REPO_NAME = 'KLTN-Token-Merging'
FORCE_FRESH_CLONE = True
ON_KAGGLE = Path('/kaggle').exists()
WORK_ROOT = Path('/kaggle/working') if ON_KAGGLE else Path.cwd()
REPO_DIR = WORK_ROOT / REPO_NAME

# ── INPUT ────────────────────────────────────────────────────────────────────
DATA_INPUT_DIR = '/kaggle/input/datasets/tuyennguyen21pt/dataset-apc/converted_apc'
DATA_INPUT_FALLBACKS = [
    '/kaggle/input/dataset-apc/converted_apc',
    '/kaggle/input/dataset-apc',
    '/kaggle/input/converted_apc',
]
CLEAN_DATA = True          # làm sạch emoji / ký tự điều khiển trước khi train

# ── OUTPUT ───────────────────────────────────────────────────────────────────
OUT = WORK_ROOT / 'outputs_ate_debug'
CKPT_DIR = OUT / 'checkpoints'
for _d in (OUT, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ── Cấu hình train ───────────────────────────────────────────────────────────
SEED = 42
ATE_MODEL_NAME = 'google/mt5-base'
OPTIMIZER = 'adafactor'    # 'adafactor' (gốc của T5, ~0 GB state) | 'adamw'
# 1063 step/epoch, mt5-base fp32 tren T4 = ~1000s train + ~40s eval
# => ~17,5 phut/epoch. 15 epoch ~ 4,4 gio, vua session 9 gio cua Kaggle.
# 30 epoch la ~8,8 gio, gan nhu chac chan bi cat giua duong.
EPOCHS = 15
BATCH_SIZE = 8
PATIENCE = 6               # 0 = tắt early stopping
LR = None                  # None = tự chọn: adafactor 1e-3, adamw 3e-4
MAX_INPUT_LEN = 128
MAX_TARGET_LEN = 64
EVAL_BATCH = 32
SKIP_TRAINING = False      # True = chỉ chạy các cell chẩn đoán không cần train

if LR is None:
    LR = 1e-3 if OPTIMIZER == 'adafactor' else 3e-4

CACHE_DIR = Path('/kaggle/temp/hf') if ON_KAGGLE else WORK_ROOT / '.hf'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(CACHE_DIR)
os.environ['TRANSFORMERS_CACHE'] = str(CACHE_DIR)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MPLBACKEND'] = 'Agg'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'


def sh(*args, cwd=None, check=True):
    cmd = [str(x) for x in args]
    print('$ ' + ' '.join(cmd), flush=True)
    return subprocess.run(cmd, cwd=cwd, check=check).returncode


if FORCE_FRESH_CLONE and REPO_DIR.exists():
    os.chdir(WORK_ROOT)
    print('Xoá repo cũ:', REPO_DIR, flush=True)
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    for _name in [m for m in sys.modules
                  if m.split('.')[0] in ('common', 'src', 'models', 'gas')]:
        del sys.modules[_name]

if (REPO_DIR / '.git').is_dir():
    sh('git', 'fetch', '--all', '--prune', cwd=REPO_DIR)
    sh('git', 'reset', '--hard', f'origin/{BRANCH}', cwd=REPO_DIR)
else:
    sh('git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_DIR)
os.chdir(REPO_DIR)
if str(REPO_DIR) in sys.path:
    sys.path.remove(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR))

print()
print('Commit đang chạy:')
sh('git', 'log', '-1', '--format=%h  %ad  %s', '--date=format:%Y-%m-%d %H:%M', cwd=REPO_DIR)
print()
print(f'Model     : {ATE_MODEL_NAME}')
print(f'Optimizer : {OPTIMIZER}   lr={LR}')
print(f'Epochs    : {EPOCHS}   batch={BATCH_SIZE}   patience={PATIENCE}')
print(f'Output    : {OUT}')

In [ ]:
# Tìm dataset và làm sạch. Giống notebook chính để hai bên so sánh được.
import re
import unicodedata

REQUIRED_FILES = ('train.apc', 'dev.apc', 'test.apc')
_NOISE_CATEGORIES = {'So', 'Sk', 'Me', 'Cc', 'Cf', 'Co', 'Cs'}


def _is_noise(ch):
    if 0xFE00 <= ord(ch) <= 0xFE0F:
        return True
    return unicodedata.category(ch) in _NOISE_CATEGORIES


def clean_line(text):
    # Không loại category 'Mn': dấu tiếng Việt dạng NFD nằm trong đó.
    text = unicodedata.normalize('NFC', text)
    text = ''.join(' ' if _is_noise(ch) else ch for ch in text)
    return re.sub(r'\s+', ' ', text).strip()


def read_apc_blocks(path):
    lines = Path(path).read_text(encoding='utf-8').splitlines()
    blocks, i = [], 0
    while i + 3 < len(lines):
        blocks.append(lines[i:i + 4])
        i += 4
    return blocks


def has_all_apc(d):
    return all((Path(d) / n).is_file() for n in REQUIRED_FILES)


DATA_DIR = None
for cand in [DATA_INPUT_DIR, *DATA_INPUT_FALLBACKS]:
    if cand and has_all_apc(cand):
        DATA_DIR = Path(cand)
        break
if DATA_DIR is None:
    root = Path('/kaggle/input')
    found = sorted(p.parent for p in root.rglob('train.apc') if has_all_apc(p.parent)) if root.is_dir() else []
    if found:
        DATA_DIR = found[0]
    elif has_all_apc(REPO_DIR / 'dataset'):
        DATA_DIR = REPO_DIR / 'dataset'
    else:
        raise FileNotFoundError('Không tìm thấy train/dev/test.apc')
print('Dataset:', DATA_DIR)

dirty_total = 0
for name in REQUIRED_FILES:
    blocks = read_apc_blocks(DATA_DIR / name)
    d = sum(1 for s, t, c, x in blocks
            if clean_line(s) != s.strip() or clean_line(t) != t.strip())
    dirty_total += d
    print(f'  {name:<10} {len(blocks):5d} mẫu, {d:5d} dòng có emoji/ký tự điều khiển')

if CLEAN_DATA and dirty_total:
    CLEAN_DIR = WORK_ROOT / 'dataset_clean'
    CLEAN_DIR.mkdir(parents=True, exist_ok=True)
    for name in REQUIRED_FILES:
        out = []
        for s, t, c, x in read_apc_blocks(DATA_DIR / name):
            cs, ct = clean_line(s), clean_line(t)
            if not cs or not x.strip():
                continue
            out.extend([cs, ct, c.strip(), x.strip().lower()])
        (CLEAN_DIR / name).write_text('\n'.join(out) + '\n', encoding='utf-8')
    DATA_DIR = CLEAN_DIR
    print('Đã làm sạch ->', DATA_DIR)
else:
    print('Không cần làm sạch.' if not dirty_total else 'CLEAN_DATA=False, giữ nguyên.')

In [ ]:
# [3] Khảo sát dữ liệu — không cần train.
from collections import Counter, defaultdict

from common.ate_dataset_utils import load_ate_records
from common.dataset_utils import parse_apc_file

records = {s: load_ate_records(DATA_DIR / f'{s}.apc') for s in ('train', 'dev', 'test')}

print()
print('=' * 78)
for split, recs in records.items():
    n_aspect = sum(len(r['aspects']) for r in recs)
    per_sent = Counter(len(r['aspects']) for r in recs)
    empty = per_sent.get(0, 0)
    words = [len(str(a).split()) for r in recs for a in r['aspects']]
    wlen = Counter(words)
    # aspect có khớp nguyên văn trong câu không (ảnh hưởng trực tiếp tới
    # build_ngram_vocabulary và LCF span)
    not_sub = sum(1 for r in recs for a in r['aspects'] if str(a) not in str(r['input_text']))
    print(f'[{split}] {len(recs)} câu, {n_aspect} aspect')
    print(f'   aspect/câu     : {dict(sorted(per_sent.items()))}   (câu không có aspect: {empty})')
    print(f'   số từ của aspect: {dict(sorted(wlen.items()))}')
    print(f'   aspect KHÔNG khớp nguyên văn trong câu: {not_sub}')

# Trùng lặp giữa train và test — nếu cao thì bài toán dễ hơn thực tế
train_terms = {str(a).lower() for r in records['train'] for a in r['aspects']}
test_terms = [str(a).lower() for r in records['test'] for a in r['aspects']]
seen = sum(1 for t in test_terms if t in train_terms)
print()
print(f'Aspect của test đã xuất hiện trong train: {seen}/{len(test_terms)} '
      f'({seen / max(len(test_terms), 1) * 100:.1f}%)')
print(f'Số aspect khác nhau trong train: {len(train_terms)}')

train_sents = {str(r['input_text']) for r in records['train']}
dup = sum(1 for r in records['test'] if str(r['input_text']) in train_sents)
print(f'Câu test trùng y nguyên với train: {dup}/{len(records["test"])}')

# Nhãn category / sentiment
for split in ('train', 'test'):
    rows = parse_apc_file(str(DATA_DIR / f'{split}.apc'))
    print()
    print(f'[{split}] category: {dict(Counter(r["aspect_category"] for r in rows).most_common())}')
    print(f'[{split}] sentiment: {dict(Counter(r["sentiment"] for r in rows).most_common())}')

In [ ]:
# [4] TRẦN CỦA PIPELINE — không cần train. Cell quan trọng nhất.
#
# Đưa chính GOLD target_text qua đúng đường mà dự đoán của model đi qua:
#     target_text -> decode_target_text -> normalize_aspects -> evaluate
# Nếu không lấy lại được 100% gold thì đó là trần cứng: model giỏi cỡ nào
# cũng không vượt được, và F1 thấp KHÔNG phải lỗi của model.
from src.metrics import evaluate_exact_match, format_metrics
from src.normalization import decode_target_text, decode_and_normalize

for split in ('dev', 'test'):
    recs = records[split]
    golds = [[str(a) for a in r['aspects']] for r in recs]
    targets = [str(r['target_text']) for r in recs]
    sents = [str(r['input_text']) for r in recs]

    # (a) chỉ decode, không normalize
    p_decode = [decode_target_text(t) for t in targets]
    m_decode = evaluate_exact_match(p_decode, golds)

    # (b) decode + normalize (đường thật của model)
    p_norm = [decode_and_normalize(t, s) for t, s in zip(targets, sents)]
    m_norm = evaluate_exact_match(p_norm, golds)

    print(f'[{split}] gold -> decode            : {format_metrics(m_decode)}')
    print(f'[{split}] gold -> decode + normalize: {format_metrics(m_norm)}   <- TRẦN THẬT')

    # liệt kê chỗ normalize làm hỏng gold
    broken = []
    for s, g, pn in zip(sents, golds, p_norm):
        if set(g) != set(pn):
            broken.append((s, g, pn))
    print(f'   số câu bị normalize làm sai lệch: {len(broken)}/{len(recs)}')
    for s, g, pn in broken[:8]:
        print(f'     câu  : {s[:88]}')
        print(f'     gold : {sorted(g)}')
        print(f'     sau  : {sorted(pn)}')
    print()

print('Đọc kết quả:')
print('  TRẦN ~1.00        -> pipeline lành, F1 thấp là do model/train.')
print('  TRẦN < 1.00 đáng kể -> normalize/metric đang ăn điểm; sửa chỗ đó')
print('                        trước khi đổi model hay tăng epoch.')

In [ ]:
# [5] Baseline từ vựng — không cần train.
#
# Với mỗi câu test, dự đoán mọi aspect đã thấy trong train mà xuất hiện trong
# câu (ưu tiên cụm dài, không chồng lấn). Đây là mức mà một bảng tra cứu đạt
# được. Nếu model sau khi train mà không hơn baseline này thì nó chưa học
# được gì ngoài ghi nhớ từ vựng.
def lexical_predict(sentence, vocab_sorted):
    low = sentence.lower()
    taken = []
    used = [False] * len(low)
    for term in vocab_sorted:
        start = low.find(term)
        while start != -1:
            end = start + len(term)
            if not any(used[start:end]):
                for i in range(start, end):
                    used[i] = True
                taken.append(sentence[start:end])
                break
            start = low.find(term, start + 1)
    return taken


vocab_sorted = sorted(train_terms, key=len, reverse=True)
sents_test = [str(r['input_text']) for r in records['test']]
golds_test = [[str(a) for a in r['aspects']] for r in records['test']]

p_lex = [lexical_predict(s, vocab_sorted) for s in sents_test]
m_lex = evaluate_exact_match(p_lex, golds_test)
print(f'baseline tra từ vựng : {format_metrics(m_lex)}')

# Baseline "đoán cả câu" và "không đoán gì" để có sàn
m_none = evaluate_exact_match([[] for _ in sents_test], golds_test)
m_all = evaluate_exact_match([[s] for s in sents_test], golds_test)
print(f'baseline không đoán gì: {format_metrics(m_none)}')
print(f'baseline đoán cả câu  : {format_metrics(m_all)}')
print()
print('Mốc để so: model sau khi train PHẢI vượt baseline tra từ vựng.')

In [ ]:
# [6] Train ATE.
import gc
import random

import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import get_constant_schedule, get_linear_schedule_with_warmup

from common.ate_dataset_utils import ATEDataset, build_tokenizer
from src.model import T5AspectExtractor
from src.trainer import ATETrainer, pick_amp_dtype


def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


def make_optimizer(params, name, lr):
    params = list(params)
    if name == 'adafactor':
        # Adafactor: optimizer goc dung de train T5/mT5. State gan nhu bang 0
        # (phan ra bac hai theo hang/cot) thay vi 8 byte/tham so nhu AdamW.
        try:
            from transformers.optimization import Adafactor
            opt = Adafactor(params, lr=lr, scale_parameter=False,
                            relative_step=False, warmup_init=False)
            print('[optim] transformers Adafactor  lr =', lr)
            return opt, 'constant'
        except Exception as exc:
            print('[optim] transformers Adafactor loi:', exc)
        opt = torch.optim.Adafactor(params, lr=lr)
        print('[optim] torch.optim.Adafactor  lr =', lr)
        return opt, 'constant'
    try:
        opt = torch.optim.AdamW(params, lr=lr, fused=True)
        print('[optim] AdamW fused=True  lr =', lr)
    except (RuntimeError, ValueError, TypeError):
        opt = torch.optim.AdamW(params, lr=lr, foreach=False)
        print('[optim] AdamW foreach=False  lr =', lr)
    return opt, 'linear'


if SKIP_TRAINING:
    print('SKIP_TRAINING=True - bo qua cell nay.')
else:
    set_seed(SEED)
    tokenizer = build_tokenizer(ATE_MODEL_NAME)
    loaders = {}
    for split in ('train', 'dev', 'test'):
        ds = ATEDataset(records[split], tokenizer,
                        max_input_length=MAX_INPUT_LEN,
                        max_target_length=MAX_TARGET_LEN)
        loaders[split] = DataLoader(ds, batch_size=BATCH_SIZE,
                                    shuffle=(split == 'train'))

    model = T5AspectExtractor(model_name=ATE_MODEL_NAME)
    model.tokenizer = tokenizer

    trainer = ATETrainer(
        model=model,
        train_loader=loaders['train'],
        dev_loader=loaders['dev'],
        test_loader=loaders['test'],
        learning_rate=LR,
        num_epochs=EPOCHS,
        output_dir=str(CKPT_DIR),
        dev_records=records['dev'],
        test_records=records['test'],
        patience=PATIENCE,
    )

    # Thay optimizer + scheduler, vi ATETrainer luon dung AdamW.
    opt, sched_kind = make_optimizer(model.model.parameters(), OPTIMIZER, LR)
    trainer.optimizer = opt
    total_steps = max(1, len(loaders['train']) * EPOCHS)
    if sched_kind == 'constant':
        trainer.scheduler = get_constant_schedule(opt)
    else:
        trainer.scheduler = get_linear_schedule_with_warmup(
            opt, num_warmup_steps=int(total_steps * 0.1),
            num_training_steps=total_steps)
    # GradScaler chi co y nghia voi fp16; pick_amp_dtype tra None tren T4.
    trainer.scaler = torch.amp.GradScaler('cuda', enabled=False)

    print('[AMP] dtype =', pick_amp_dtype() or 'fp32 (tat autocast)')
    if torch.cuda.is_available():
        free_b, total_b = torch.cuda.mem_get_info()
        print(f'[VRAM] trong {free_b / 2**30:.2f} / {total_b / 2**30:.2f} GB')
    print(f'[data] {len(loaders["train"].dataset)} cau train, '
          f'{len(loaders["train"])} step/epoch')
    print()

    result = trainer.train(eval_test=False)
    print()
    print('best dev F1    :', result['best_dev_f1'])
    print('best checkpoint:', result['best_checkpoint'])
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# [7] Model sinh ra CHU GI - va ghi file du doan.
#
# In nguyen van output cua model roi lan theo tung buoc bien doi:
#     raw -> decode_target_text -> normalize_aspects -> so voi gold
# Ba kha nang, moi kha nang can mot cach sua khac nhau:
#   raw sai dinh dang (khong co dau ngoac) -> model chua hoc duoc format
#   raw dung nhung decode rong             -> loi regex / dinh dang target
#   decode dung nhung normalize lech       -> loi n-gram (xem cell 4)
import csv

from src.inference import predict_aspects_for_records

if SKIP_TRAINING:
    BEST = CKPT_DIR / 'best'
else:
    BEST = Path(result['best_checkpoint'] or (CKPT_DIR / 'best'))
print('Nap checkpoint:', BEST)
_dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
eval_model = T5AspectExtractor.from_pretrained(str(BEST), device=_dev)

recs_test = records['test']
sents_test = [str(r['input_text']) for r in recs_test]
golds_test = [[str(a) for a in r['aspects']] for r in recs_test]

print()
print('=' * 78)
print('RAW OUTPUT (15 cau dau)')
print('=' * 78)
for r in recs_test[:15]:
    sent = str(r['input_text'])
    enc = eval_model.tokenizer(sent, max_length=MAX_INPUT_LEN,
                               padding='max_length', truncation=True,
                               return_tensors='pt')
    ids = eval_model.generate(enc['input_ids'], attention_mask=enc['attention_mask'])
    raw = eval_model.tokenizer.decode(ids[0], skip_special_tokens=True)
    dec = decode_target_text(raw)
    nor = decode_and_normalize(raw, sent)
    ok = set(nor) == {str(a) for a in r['aspects']}
    print(('OK  ' if ok else 'SAI ') + 'cau      : ' + sent[:86])
    print('     raw      : ' + repr(raw))
    print('     decode   : ' + str(dec))
    print('     normalize: ' + str(nor))
    print('     gold     : ' + str(sorted(str(a) for a in r['aspects'])))

print()
print('Dang du doan toan bo tap test...', flush=True)
preds_test, _ = predict_aspects_for_records(
    eval_model, recs_test, max_input_length=MAX_INPUT_LEN, batch_size=EVAL_BATCH)
m_test = evaluate_exact_match(preds_test, golds_test)
print('TEST                       :', format_metrics(m_test))
print('baseline tra tu vung       :', format_metrics(m_lex))
print('tran pipeline (cell 4)     :', format_metrics(m_norm))

# (1) test_predictions.csv - dinh dang ma giai doan APC can
pred_csv = OUT / 'test_predictions.csv'
rows = []
for r, ps in zip(recs_test, preds_test):
    sent = str(r['input_text'])
    gold_str = '|'.join(str(a) for a in r['aspects'])
    if ps:
        for p in ps:
            rows.append({'sentence': sent, 'predicted_term': p, 'gold_terms': gold_str})
    else:
        rows.append({'sentence': sent, 'predicted_term': '', 'gold_terms': gold_str})
with open(pred_csv, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['sentence', 'predicted_term', 'gold_terms'])
    w.writeheader()
    w.writerows(rows)
print()
print(f'{pred_csv}  ({len(rows)} dong)')

# (2) test_pred.apc - dinh dang .apc 4 dong, dat lai $T$ vao cau.
#     ATE CHI du doan aspect term, KHONG du doan category/sentiment nen hai
#     dong do la cho giu cho. File nay de soi bang mat, khong phai nhan vang.
pred_apc = OUT / 'test_pred.apc'
blocks = []
for r, ps in zip(recs_test, preds_test):
    sent = str(r['input_text'])
    for p in ps:
        idx = sent.find(p)
        masked = sent[:idx] + '$T$' + sent[idx + len(p):] if idx >= 0 else sent + ' $T$'
        blocks.extend([masked, p, 'UNKNOWN', 'neutral'])
pred_apc.write_text('\n'.join(blocks) + '\n', encoding='utf-8')
print(f'{pred_apc}  ({len(blocks) // 4} mau)  [category/sentiment la cho giu cho]')

# (3) test_ate_compare.csv - so sanh canh nhau
cmp_csv = OUT / 'test_ate_compare.csv'
with open(cmp_csv, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['sentence', 'gold', 'pred', 'ket_qua'])
    for sent, g, p in zip(sents_test, golds_test, preds_test):
        sg, sp = set(g), set(p)
        if sg == sp:
            verdict = 'dung'
        elif not sp:
            verdict = 'khong doan'
        elif sp & sg:
            verdict = 'dung mot phan'
        else:
            verdict = 'sai'
        w.writerow([sent, ' | '.join(sorted(sg)), ' | '.join(sorted(sp)), verdict])
print(f'{cmp_csv}')

In [ ]:
# [7b] Evaluation tương thích với experiments/run_ate_inference.py.
#
# File .py đánh giá từng dòng gold aspect trong test_sentences_id.csv:
# strip -> lower -> bỏ dấu chấm, sau đó kiểm tra gold có nằm trong
# danh sách prediction của câu hay không. Ở đây dùng các dòng .apc tương ứng.
from common.ate_dataset_utils import parse_apc_file_for_ate
from src.metrics import compute_prf


def normalize_eval_term(term):
    return str(term).strip().lower().replace('.', '')


raw_test_rows = parse_apc_file_for_ate(DATA_DIR / 'test.apc')
preds_by_sentence = {
    sentence: [normalize_eval_term(term) for term in preds]
    for sentence, preds in zip(sents_test, preds_test)
}

total_tp = total_fp = total_fn = 0
for row in raw_test_rows:
    sentence = str(row['text'])
    gold = normalize_eval_term(row['aspect_term'])
    pred_set = set(preds_by_sentence.get(sentence, []))
    tp = int(gold in pred_set)
    total_tp += tp
    total_fp += len(pred_set) - tp
    total_fn += 1 - tp

m_filepy = compute_prf(total_tp, total_fp, total_fn)
m_filepy.update({
    'tp': float(total_tp),
    'fp': float(total_fp),
    'fn': float(total_fn),
})
print('EVALUATION THEO experiments/run_ate_inference.py')
print('test raw records             :', len(raw_test_rows))
print('P/R/F1 (lower + trim + dot)  :', format_metrics(m_filepy))
print('exact grouped-set F1          :', format_metrics(m_test))


In [ ]:
# [8] Phan loai loi va ket luan.
from collections import Counter

kinds = Counter()
examples = {}
empty_sents = 0

for sent, golds, preds in zip(sents_test, golds_test, preds_test):
    if not preds:
        empty_sents += 1
    for p in preds:
        if p in golds:
            k = 'dung hoan toan'
        elif any(p.lower() == g.lower() for g in golds):
            k = 'chi khac chu hoa/thuong'
        elif any(p in g or g in p for g in golds):
            k = 'lech bien (chua nhau)'
        elif any(set(p.lower().split()) & set(g.lower().split()) for g in golds):
            k = 'trung mot phan tu'
        else:
            k = 'khong lien quan'
        kinds[k] += 1
        examples.setdefault(k, []).append((sent, p, sorted(golds)))

n_pred = sum(kinds.values())
n_gold = sum(len(g) for g in golds_test)
n_sent = len(sents_test)
print(f'cau test           : {n_sent}')
print(f'aspect gold        : {n_gold}')
print(f'aspect du doan     : {n_pred}')
print(f'cau khong doan gi  : {empty_sents} ({empty_sents / max(n_sent, 1) * 100:.1f}%)')
print(f'ti le sinh/gold    : {n_pred / max(n_gold, 1):.2f}   (<1 = sinh thieu)')
print()
for k, c in kinds.most_common():
    print(f'  {k:<26} {c:6d}  ({c / max(n_pred, 1) * 100:5.1f}%)')

near = sum(kinds[k] for k in ('lech bien (chua nhau)',
                              'chi khac chu hoa/thuong',
                              'trung mot phan tu'))
print()
print('=' * 78)
print('KET LUAN')
print('=' * 78)
print(f'tran pipeline (cell 4)        F1 = {m_norm["f1"]:.4f}')
print(f'baseline tra tu vung (cell 5) F1 = {m_lex["f1"]:.4f}')
print(f'model sau khi train           F1 = {m_test["f1"]:.4f}')
print()
if m_norm['f1'] < 0.95:
    print('[!] Tran pipeline duoi 0.95 -> normalize/metric dang an diem.')
    print('    Sua cho do TRUOC; doi model hay tang epoch khong cuu duoc.')
if m_test['f1'] < m_lex['f1']:
    print('[!] Model KHONG hon baseline tra tu vung -> chua hoc duoc gi them.')
    print('    Nghi: train chua du epoch, lr sai, hoac model qua nho.')
if near > kinds['khong lien quan']:
    print('[i] Loi chu yeu la GAN DUNG (lech bien) -> van de quy uoc span,')
    print('    khong phai dung luong model. Xem lai cach gan aspect trong data.')
elif kinds['khong lien quan'] > near:
    print('[i] Loi chu yeu KHONG LIEN QUAN -> model that su chua hoc duoc.')
    print('    Tang epoch / doi model lon hon la huong dung.')
if empty_sents / max(n_sent, 1) > 0.3:
    print('[i] Tren 30% cau khong doan gi -> model sinh "none" qua nhieu.')
    print('    Kiem tra ti le cau khong co aspect trong train (cell 3).')

for k in ('lech bien (chua nhau)', 'khong lien quan'):
    if examples.get(k):
        print()
        print('--- vi du: ' + k + ' ---')
        for sent, p, g in examples[k][:8]:
            print('  cau  : ' + sent[:86])
            print('  doan : ' + repr(p))
            print('  gold : ' + str(g))